# Data Preparation and Validation

## Objective
Prepare the five raw RavenStack SaaS datasets for SQL analysis,
Excel reporting, and Power BI dashboards.

## Preparation Strategy
- Preserve the original raw datasets.
- Standardize date columns into consistent datetime formats.
- Validate data types, identifiers, and relationships.
- Investigate known data-quality issues without deleting records automatically.
- Preserve legitimate missing values.
- Export validated datasets to the `data/processed/` directory.
- Verify the processed datasets against the original datasets.

## Datasets
1. Accounts
2. Subscriptions
3. Feature Usage
4. Support Tickets
5. Churn Events

Establish the file paths and confirm that the five raw datasets can be accessed

In [1]:
from pathlib import Path
import pandas as pd

PROJECT_DRI = Path('..')
RAW_DRI=PROJECT_DRI / 'data' / 'raw'
PROCESSED_DIR=PROJECT_DRI / 'data'/'processed'

PROCESSED_DIR.mkdir(parents=True,exist_ok=True)

datasets = {
    "accounts": "ravenstack_accounts.csv",
    "subscriptions": "ravenstack_subscriptions.csv",
    "feature_usage": "ravenstack_feature_usage.csv",
    "support_tickets": "ravenstack_support_tickets.csv",
    "churn_events": "ravenstack_churn_events.csv",
}

for name, filename in datasets.items():
    file_path= RAW_DRI / filename
    
    print(f"{name}: {'FOUND' if file_path.exists() else 'NOT FOUND'}")

accounts: FOUND
subscriptions: FOUND
feature_usage: FOUND
support_tickets: FOUND
churn_events: FOUND


Load datasets

In [2]:
data={}

for name, filename in datasets.items():
    file_path=RAW_DRI/filename
    
    data[name]=pd.read_csv(file_path)
    
    print(f"{name:} {data[name].shape}")

accounts (500, 10)
subscriptions (5000, 14)
feature_usage (25000, 8)
support_tickets (2000, 9)
churn_events (600, 9)


In [3]:
for name,df in data.items():
    data_columns=[
        column for column in df.columns
        if 'date' in column.lower() or 'at' in column.lower()
    ]

    print(f"{name:} {data_columns}")

accounts ['signup_date', 'seats']
subscriptions ['start_date', 'end_date', 'seats']
feature_usage ['usage_date', 'feature_name', 'usage_duration_secs', 'is_beta_feature']
support_tickets ['submitted_at', 'closed_at', 'satisfaction_score', 'escalation_flag']
churn_events ['churn_date', 'is_reactivation']


In [4]:
date_columns = {
    "accounts": ["signup_date"],
    "subscriptions": ["start_date", "end_date"],
    "feature_usage": ["usage_date"],
    "support_tickets": ["submitted_at", "closed_at"],
    "churn_events": ["churn_date"]
}

# Display the selected columns
for name, columns in date_columns.items():
    print(f"{name}: {columns}")

accounts: ['signup_date']
subscriptions: ['start_date', 'end_date']
feature_usage: ['usage_date']
support_tickets: ['submitted_at', 'closed_at']
churn_events: ['churn_date']


Convert the date columns

In [6]:
for name, columns in date_columns.items():
    for column in columns:
        data[name][column] = pd.to_datetime(
            data[name][column],
            errors='coerce'
        )
print("Date conversion completed")

Date conversion completed


Verify date conversions

In [7]:
for name,columns in date_columns.items():
    for column in columns:
        print(
            f"{column} {data[name][column].dtype} | "
            f"Missing values: {data[name][column].isna().sum()}"
        )

signup_date datetime64[us] | Missing values: 0
start_date datetime64[us] | Missing values: 0
end_date datetime64[us] | Missing values: 4514
usage_date datetime64[us] | Missing values: 0
submitted_at datetime64[us] | Missing values: 0
closed_at datetime64[us] | Missing values: 0
churn_date datetime64[us] | Missing values: 0


Save the processed datasets

In [8]:
for name, filename in datasets.items():
    output_path=PROCESSED_DIR/filename
    
    data[name].to_csv(output_path)
    
    print(f"Saved: {output_path}")

Saved: ..\data\processed\ravenstack_accounts.csv
Saved: ..\data\processed\ravenstack_subscriptions.csv
Saved: ..\data\processed\ravenstack_feature_usage.csv
Saved: ..\data\processed\ravenstack_support_tickets.csv
Saved: ..\data\processed\ravenstack_churn_events.csv


Validate the saved files

In [13]:
for name,filename in datasets.items():
    processed_df=pd.read_csv(PROCESSED_DIR/filename)
    original_df=pd.read_csv(RAW_DRI/filename)
    
    print(
        f"{name}: "
        f"Rows preserved = {len(processed_df)==len(original_df)} | "
        f"Columns preserved = {list(processed_df.columns)==list(processed_df.columns)}"
    )

accounts: Rows preserved = True | Columns preserved = True
subscriptions: Rows preserved = True | Columns preserved = True
feature_usage: Rows preserved = True | Columns preserved = True
support_tickets: Rows preserved = True | Columns preserved = True
churn_events: Rows preserved = True | Columns preserved = True
